# Lab 3: Post-Training Vision Language Models

## Cosmos 3 Nano + LoRA

In **Lab 1** you deployed Cosmos 3 Reasoner and learned to prompt it - and saw that prompting
alone has a ceiling. In the **SDG notebook** you generated synthetic traffic footage to fill
the gaps in the dataset. This notebook closes the loop: you will **post-train** Cosmos 3 Nano
on that footage with **LoRA**, and measure exactly what training bought you on clips the model
has never seen.

The task is the one the whole day builds toward - spotting traffic anomalies on a fixed
camera. The model answers one line per clip:

```
ANOMALY=<yes|no>; TYPE=<collision|stalled_vehicle|road_obstruction|wrong_way|speeding|none>
```

### Learning Objectives:
This notebook explores the following topics:
* Why post-training is needed once prompt engineering runs out of room
* How to organise a video dataset for VLM fine-tuning, and how to split it without leaking
* Why **macro-F1** is the right metric for an imbalanced detection task
* Establishing a zero-shot baseline *before* training anything
* **LoRA** fine-tuning with NVIDIA TAO and Cosmos-RL, and the spec keys that matter
* Reading validation loss vs. real task accuracy - and why they disagree
* Evaluating the tuned checkpoint on held-out clips and comparing per class
* What the per-class result tells you about which data to generate next

> **Runtime:** about 20 minutes end to end. Training is ~3 minutes; the two evaluations are
> the slower parts.

### Table of Contents

**[Set Up the Environment](#Set-Up-the-Environment)**  
**[Part 1: Why Post-Training?](#Part-1:-Why-Post-Training?)**  
**[Part 2: Prepare the Dataset](#Part-2:-Prepare-the-Dataset)**  
&nbsp;&nbsp;&nbsp;&nbsp;[2.1 The Source Footage](#2.1-The-Source-Footage)  
&nbsp;&nbsp;&nbsp;&nbsp;[2.2 How Many Distinct Clips?](#2.2-How-Many-Distinct-Clips?)  
&nbsp;&nbsp;&nbsp;&nbsp;[2.3 Split and Cut](#2.3-Split-and-Cut)  
&nbsp;&nbsp;&nbsp;&nbsp;[2.4 The Annotation Format](#2.4-The-Annotation-Format)  
**[Part 3: Zero-Shot Evaluation](#Part-3:-Zero-Shot-Evaluation)**  
**[Part 4: LoRA Fine-Tuning](#Part-4:-LoRA-Fine-Tuning)**  
&nbsp;&nbsp;&nbsp;&nbsp;[4.1 The Training Spec](#4.1-The-Training-Spec)  
&nbsp;&nbsp;&nbsp;&nbsp;[4.2 Train](#4.2-Train)  
&nbsp;&nbsp;&nbsp;&nbsp;[4.3 Validation Loss](#4.3-Validation-Loss)  
**[Part 5: Evaluate the Fine-Tuned Model](#Part-5:-Evaluate-the-Fine-Tuned-Model)**  
**[Part 6: Compare](#Part-6:-Compare)**  
**[Part 7: Tear Down](#Part-7:-Tear-Down)**  
**[Review](#Review)**

### Set Up the Environment

#### Credentials

Two tokens are needed. **Set them in your shell before launching Jupyter** — never paste
secrets into a notebook cell that gets saved.

```bash
export NGC_KEY=...     # nvcr.io container images
export HF_TOKEN=...    # gated Cosmos3 model repos on HuggingFace
```

The next cell only checks that they are *present*; it never prints their values.

In [ ]:
import os, sys, subprocess, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import dli_helpers as H

print("Credentials")
have_ngc = H.check("NGC_KEY set",  bool(os.environ.get("NGC_KEY")),  "needed only if the TAO image must be pulled")
have_hf  = H.check("HF_TOKEN set", bool(os.environ.get("HF_TOKEN")), "needed only if the base checkpoint must be downloaded")

#### GPU

This notebook uses **one GPU** for training and evaluation. Cosmos 3 Nano LoRA needs
roughly **60 GB**, so an 80 GB card (H100 / A100-80G) is the target.

In [ ]:
gpus = H.gpu_table()
for g in gpus:
    print(f"  GPU {g['index']}: {g['name']}  {g['used_gb']:.0f} / {g['total_gb']:.0f} GB used")

# Pick the GPU this notebook will use. Override if you are sharing a node.
GPU = int(os.environ.get("DLI_GPU", gpus[0]["index"]))
print(f"\nUsing GPU {GPU}")

free_gb = gpus[GPU]["total_gb"] - gpus[GPU]["used_gb"]
H.check(f"GPU {GPU} has >= 60 GB free", free_gb >= 60, f"{free_gb:.0f} GB free")

#### Docker and the TAO container

Training runs inside the TAO Cosmos-RL container. If the image is missing, pull it
(~28 GB — normally pre-staged on the DLI instance).

In [ ]:
ok_docker = subprocess.run(["docker", "info"], capture_output=True).returncode == 0
H.check("docker daemon reachable", ok_docker)

import shutil
free_disk = shutil.disk_usage(Path.cwd()).free / 1e9
H.check("disk >= 150 GB free", free_disk >= 150,
        f"{free_disk:.0f} GB — training keeps 2 FSDP checkpoints (~66 GB each)")

img = subprocess.run(["docker", "images", "-q", H.IMAGE], capture_output=True, text=True).stdout.strip()
if not H.check(f"TAO image present ({H.IMAGE})", bool(img)):
    print("\n  Pull it with:")
    print(f"    docker login nvcr.io -u '$oauthtoken' -p $NGC_KEY")
    print(f"    docker pull {H.IMAGE}")

#### Base checkpoint

The TAO 7.0.1 image cannot load the native `cosmos3_omni` checkpoint format, so training uses
a **Qwen3-VL safetensors** conversion of Cosmos 3 Nano. On the DLI instance this is
pre-staged and the next cell simply verifies it.

If you need to create it yourself, the conversion script ships in the **TAO Skill Bank**.
This is a *setup* step (~15 min) - not something to run live:

```bash
# 1. clone the skill bank (contains the conversion script)
git clone https://github.com/NVIDIA-TAO/tao-skill-bank.git
cd tao-skill-bank

# 2. convert the native Cosmos3-Nano checkpoint to Qwen3-VL safetensors
python skills/models/tao-finetune-cosmos-reason/scripts/prepare_cosmos3_vlm_checkpoint.py \
    --checkpoint-path /path/to/Cosmos3-Nano \
    --output-path     /path/to/Cosmos3-Nano-VLM
```

The same repo holds the `tao-finetune-cosmos-reason` skill used by the agentic workflow in
Part 2 of the day, so it is worth having locally either way.

In [ ]:
# Defaults assume the DLI layout; override with DLI_PTM / DLI_DATA if yours differs.
PTM = Path(os.environ.get("DLI_PTM", "/workspace/models/Cosmos3-Nano-VLM"))
shards = sorted(PTM.glob("*.safetensors"))
if H.check(f"base checkpoint at {PTM}", bool(shards), f"{len(shards)} shards"):
    cfg = json.loads((PTM / "config.json").read_text())
    print(f"       model_type={cfg['model_type']}  arch={cfg['architectures'][0]}")
    print(f"       hidden={cfg.get('text_config', cfg).get('hidden_size')}  "
          f"layers={cfg.get('text_config', cfg).get('num_hidden_layers')}")
    # The container runs as your uid; unreadable shards surface as a confusing
    # "No such file or directory" much later, so check readability now.
    try:
        shards[0].open("rb").read(8)
        H.check("checkpoint readable by this user", True)
    except PermissionError:
        H.check("checkpoint readable by this user", False, "run: chmod -R u+r <ptm>")

#### Configuration

Every path and hyperparameter for the run, in one place.

In [ ]:
ROOT = Path(os.environ.get("DLI_ROOT", Path.cwd())).resolve()
DATA = Path(os.environ.get("DLI_DATA", "/workspace/data/SDG Clips"))
WORK = ROOT / "work"; WORK.mkdir(exist_ok=True)

SFT   = WORK / "sft"       # dataset the trainer reads
RUNS  = WORK / "runs"      # training output (adapters)
EVALS = WORK / "evals"     # evaluation output
LOGS  = WORK / "logs"
for d in (SFT, RUNS, EVALS, LOGS): d.mkdir(parents=True, exist_ok=True)

# ---- dataset -------------------------------------------------------------
WINDOW_S        = 8.0    # clip length shown to the model
STRIDE_ANOMALY  = 2.0    # dense sampling of the rare class
STRIDE_NORMAL   = 4.0    # normals are plentiful
NFRAMES         = 8      # frames sampled per clip (~1 fps over an 8s window)

# ---- LoRA ----------------------------------------------------------------
LORA_R          = 16
LORA_ALPHA      = 32
LORA_TARGETS    = ["q_proj", "v_proj"]
EPOCHS          = 3
BATCH           = 4      # must be a multiple of mini_batch (=1)
LR              = 1e-4

print(f"root   {ROOT}\ndata   {DATA}\nGPU    {GPU}")
print(f"\nLoRA r={LORA_R} alpha={LORA_ALPHA} targets={LORA_TARGETS} "
      f"epochs={EPOCHS} batch={BATCH} lr={LR}")

---
### Part 1: Why Post-Training?

In Lab 1 you saw Cosmos 3 answer questions about traffic video straight out of the box. So
why train it at all?

Because prompting has a ceiling, and on this task you can measure exactly where it sits. The
base model is **precise but timid**: when it flags an anomaly it is almost always right, but
it misses most of them. No amount of prompt rewriting moved that - naming the classes,
describing the camera geometry, asking for a confidence score, adding a reasoning step. The
model is not confused; it is under-calling.

That is the signature of a problem post-training fixes. Prompting changes *what you ask*;
post-training changes *what the model has seen*.

#### Full fine-tuning vs LoRA

| | Full fine-tune | **LoRA** |
|---|---|---|
| what updates | every weight | small low-rank adapters |
| optimizer state | for all 8B params | for ~0.1% of them |
| artifact | a new 17 GB model | a **15 MB** adapter |
| risk | catastrophic forgetting | base weights frozen |

LoRA inserts a pair of small matrices beside the attention projections and trains only those.
On this dataset it is also *faster* than full fine-tuning a model a quarter of the size - the
frozen base needs no gradients or optimizer moments.

#### The rule this notebook is really teaching

Watch the per-class results at the end. Fine-tuning transfers **where you gave it data, and
nowhere else**. That is what turns the SDG notebook from a nice-to-have into the thing that
decides whether the model works.

---
### Part 2: Prepare the Dataset

#### 2.1 The Source Footage

The source footage is three ~5-minute traffic recordings, each assembled from a library of
short clips. A JSON timeline says exactly which clip occupies which time range and whether
it is an anomaly — so labels come from the authoring metadata, not from guesswork.

In [ ]:
VIDEOS = ["site03_assembly_01", "site03_assembly_02", "site03_assembly_03"]

timelines = {v: json.loads((DATA / f"{v}.json").read_text()) for v in VIDEOS}
m = timelines[VIDEOS[0]]
print(f"{VIDEOS[0]}: {m['duration_seconds']:.0f}s, {m['resolution']}, {m['fps']} fps")
print(f"segments: {len(m['timeline'])}, anomalies: {len(m['anomalies'])}\n")
for a in m["anomalies"]:
    print(f"  {a['anomaly']:<18} {a['start']:7.1f} - {a['end']:7.1f}s   ({a['clip']})")

#### 2.2 How Many Distinct Clips?

This is the most important cell in the dataset section. The three recordings contain 56
timeline segments — but many are **the same source clip reused**. Training on one copy and
testing on another would leak, and the score would be meaningless.

In [ ]:
from collections import Counter
segments = [(v, s) for v in VIDEOS for s in timelines[v]["timeline"]]
names = [s["clip"] for _, s in segments]
anomaly_clips = {s["clip"]: s["anomaly"] for _, s in segments if s["kind"] == "anomaly"}

print(f"timeline segments : {len(segments)}")
print(f"unique clips      : {len(set(names))}   <-- the real dataset size")
print(f"unique anomalies  : {len(anomaly_clips)}")
print(f"\nreused clips: {[c for c, n in Counter(names).items() if n > 1][:6]} ...")
print(f"\nanomaly clips by class: {Counter(anomaly_clips.values())}")

#### 2.3 Split and Cut

Every window cut from a clip goes to the **same** side of the split, so no frame appears in
both train and validation.

`wrong_way` deliberately gets **one clip in train and a different one in validation** — the
only arrangement that can test whether the model generalizes to an unseen wrong-way clip.
`speeding` and `collision` have a single clip each, so they go to train and **cannot be
scored** on held-out data.

In [ ]:
VAL_ANOMALY   = {"wrongway_b_v2", "chain_obstruction", "chain_stalled_short"}
TRAIN_ANOMALY = {"wrongway_a", "chain_obstruction_v2", "chain_stalled",
                 "chain_collision", "site03_speeding_vehicle"}

SYSTEM_PROMPT = (
    "You are monitoring a fixed traffic camera at a suburban road intersection. "
    "The camera never moves. You detect traffic anomalies: collision, stalled_vehicle, "
    "road_obstruction, wrong_way, speeding. Normal traffic of any density, vehicles "
    "waiting at a red light, and an empty road are NOT anomalies."
)
QUESTION = ("Does this clip contain a traffic anomaly? Reply with exactly one line: "
            "ANOMALY=<yes|no>; TYPE=<collision|stalled_vehicle|road_obstruction|"
            "wrong_way|speeding|none>")

vids = SFT / "videos"; vids.mkdir(parents=True, exist_ok=True)
jobs, train, val, seen = [], [], [], set()

for v in VIDEOS:
    for seg in timelines[v]["timeline"]:
        clip = seg["clip"]
        if clip in seen:            # dedup: one copy of each source clip
            continue
        seen.add(clip)
        is_anom = seg["kind"] == "anomaly"
        cls = seg["anomaly"] or "none"
        if is_anom:
            split = "val" if clip in VAL_ANOMALY else "train"
            stride = STRIDE_ANOMALY
        else:
            split = "val" if (len(seen) % 10) in (0, 1, 2) else "train"
            stride = STRIDE_NORMAL

        for t0, t1 in H.windows(seg["start"], seg["end"], WINDOW_S, stride):
            name = f"{clip}__{t0:.1f}_{t1:.1f}.mp4"
            jobs.append((DATA / f"{v}.mp4", vids / name, t0, t1 - t0))
            answer = f"ANOMALY=yes; TYPE={cls}" if is_anom else "ANOMALY=no; TYPE=none"
            (val if split == "val" else train).append({
                "id": name[:-4], "video": f"videos/{name}",
                "conversations": [{"from": "human", "value": f"<video>\n{QUESTION}"},
                                  {"from": "gpt",   "value": answer}],
                "clip_name": clip, "anomaly_class": cls,
            })

print(f"cutting {len(jobs)} windows with ffmpeg ...")
H.cut_all(jobs)
H.write_llava(train, SFT / "train")
H.write_llava(val,   SFT / "val")
print(f"train {len(train)} windows / val {len(val)} windows")

In [ ]:
# Per-class composition — read this before trusting any later number.
rows = {}
for split, recs in (("train", train), ("val", val)):
    for r in recs:
        k = r["anomaly_class"]
        rows.setdefault(k, {"train_clips": set(), "train_w": 0, "val_clips": set(), "val_w": 0})
        rows[k][f"{split}_clips"].add(r["clip_name"]); rows[k][f"{split}_w"] += 1

print(f"{'class':<20}{'train clips':>12}{'train win':>11}{'val clips':>11}{'val win':>9}")
for k in ["road_obstruction", "stalled_vehicle", "collision", "wrong_way", "speeding", "none"]:
    if k in rows:
        r = rows[k]
        print(f"{k:<20}{len(r['train_clips']):>12}{r['train_w']:>11}"
              f"{len(r['val_clips']):>11}{r['val_w']:>9}")
print(f"\n{'TOTAL':<20}{len({r['clip_name'] for r in train}):>12}{len(train):>11}"
      f"{len({r['clip_name'] for r in val}):>11}{len(val):>9}")
print("\nNote: every anomaly class has exactly ONE training clip. That is the ceiling"
      "\non what this dataset can teach — keep it in mind when reading the results.")

#### 2.4 The Annotation Format

The trainer reads a JSON **list**; each item has a `video` path and a `conversations` pair.
This is the classic *llava* shape.

> **Gotcha worth knowing:** the same container also ships a different data hook that expects
> `{"media_root": ..., "items": [...]}`. Feed that shape here and you get a silent
> **zero-length dataset** that fails later as `training failed: 0` — a division by zero,
> with nothing pointing at the annotation file.

In [ ]:
print(json.dumps(train[0], indent=2)[:600])

---
### Part 3: Zero-Shot Evaluation

Before training, measure the **untuned** model on the held-out clips. Skipping this step is
the most common way to end up unable to say whether fine-tuning helped.

We score with **macro-F1**: the mean of the anomaly-class F1 and the normal-class F1. Plain
accuracy is misleading on an imbalanced set — a model that answers "no" to everything can
look respectable on accuracy while detecting nothing.

In [ ]:
import tomli_w

def eval_config(lora_dir_in_container=None, nframes=NFRAMES):
    # Two shapes matter here: `task` must be a DICT (the evaluator calls
    # task.get("type")) and `metrics` must be a DICT with `names`. Passing a
    # bare string or list fails with "'str'/'list' object has no attribute 'get'".
    return {
        "dataset":    {"annotation_path": "/data/val/annotations.json", "media_dir": "/data"},
        "model":      {"model_name": lora_dir_in_container or "/ptm",
                       "base_model_path": "/ptm",
                       "enable_lora": bool(lora_dir_in_container)},
        "task":       {"type": ""},
        "evaluation": {"batch_size": 1},
        "vision":     {"nframes": nframes},
        "generation": {"temperature": 0.0, "max_new_tokens": 64},
        "metrics":    {"names": ["accuracy", "f1"]},
        "results":    {}, "num_gpus": 1, "results_dir": "/results",
    }

def run_eval(name, lora_host=None):
    out = EVALS / name; out.mkdir(parents=True, exist_ok=True)
    (out / "config.toml").write_bytes(tomli_w.dumps(eval_config("/lora" if lora_host else None)).encode())
    mounts = [(PTM, "/ptm:ro"), (SFT, "/data:ro"), (out, "/results")]
    if lora_host:
        mounts.append((Path(lora_host).resolve(), "/lora:ro"))
    rc, dt = H.run_container(f"dli_eval_{name}", GPU, mounts,
                             ["cosmos-rl-evaluate", "--config", "/results/config.toml"],
                             LOGS / f"eval_{name}.log")
    items, path = H.load_eval_results(out)
    return H.score(items), dt

print("Evaluating the BASE model on the held-out clips (a few minutes)...\n")
base_metrics, base_time = run_eval("baseline")
H.show("Cosmos 3 Nano (zero-shot)", base_metrics)

---
### Part 4: LoRA Fine-Tuning

#### 4.1 The Training Spec

TAO drives Cosmos-RL from a TOML spec. The keys that matter here:

| key | value | why |
|---|---|---|
| `policy.lora.r` / `lora_alpha` | 16 / 32 | adapter capacity; alpha/r = 2× scaling |
| `policy.lora.target_modules` | `q_proj, v_proj` | which projections get adapters |
| `policy.model_max_length` | 40960 | **do not lower** — video inputs hit a `vision_embeds` shape mismatch below ~40k |
| `train.train_policy.type` | `sft` | drop it and Cosmos-RL switches to RL mode and tries to allocate a rollout replica |
| `train.train_batch_per_replica` | 4 | must be a multiple of `mini_batch` (1) |
| `train.ckpt.max_keep` | 2 | full FSDP checkpoints are ~66 GB each. **Do not set 1** — retention then deletes the directory while the async completion-marker thread is still writing into it, and the run dies with `FileNotFoundError: .rank_0_complete`. Delete them in the cleanup cell instead. |

In [ ]:
spec = {
    "train": {
        # `compile` MUST be present and False: the HFModel path asserts on it,
        # and omitting the key does NOT give you the template's default.
        "compile": False,
        "epoch": EPOCHS, "train_batch_per_replica": BATCH, "optm_lr": LR,
        "optm_impl": "foreach", "epsilon": 1e-8, "optm_betas": [0.9, 0.999],
        "optm_min_lr_factor": 0.0, "async_tp_enabled": False,
        "fsdp_reduce_dtype": "float32", "fsdp_offload": False,
        "fsdp_reshard_after_forward": "default", "sync_weight_interval": 1,
        "fp8": {"enable_fp8": False},
        "optm_name": "AdamW", "optm_weight_decay": 0.01, "optm_warmup_epochs": 0,
        "optm_decay_type": "linear", "optm_grad_norm_clip": 1.0,
        "master_dtype": "float32", "param_dtype": "bfloat16",
        "output_dir": "/results/output", "resume": False,
        "train_policy": {"type": "sft", "mini_batch": 1,
                         "conversation_column_name": "conversations",
                         "dataset": {"name": "wts", "test_size": 0}},
        "ckpt": {"enable_checkpoint": True, "save_freq_in_epoch": 1,
                 "export_safetensors": True, "max_keep": 2},
    },
    "validation": {"enable": True, "freq_in_epoch": 1, "batch_size": 1,
                   "dataloader_num_workers": 0, "enable_dataset_cache": False,
                   "dataset": {"name": "", "subset": "", "split": "train"}},
    "policy": {
        "model_name_or_path": "/ptm",
        "model_max_length": 40960,
        "model_gradient_checkpointing": True,
        "parallelism": {"dp_shard_size": 1, "dp_replicate_size": 1, "tp_size": 1,
                        "cp_size": 1, "pp_size": 1, "n_init_replicas": 1},
        "lora": {"r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": 0.0,
                 "target_modules": LORA_TARGETS, "init_lora_weights": True,
                 "use_rslora": False, "modules_to_save": []},
    },
    "custom": {
        "train_dataset": {"annotation_path": "/data/train/annotations.json",
                          "media_path": "/data", "media_root": "/data"},
        "val_dataset":   {"annotation_path": "/data/val/annotations.json",
                          "media_path": "/data", "media_root": "/data"},
        "vision": {"nframes": NFRAMES},
        "system_prompt": SYSTEM_PROMPT,
    },
    "logging": {"logger": ["console"], "project_name": "sgai-traffic"},
    "results_dir": "/results",
    "redis": "12800",
}

RUN = RUNS / "lora_r16"; RUN.mkdir(parents=True, exist_ok=True)
(RUN / "config.toml").write_bytes(tomli_w.dumps(spec).encode())
print(f"spec written to {RUN/'config.toml'}")
print(f"steps/epoch = {len(train)} samples / batch {BATCH} = {len(train)//BATCH}")

#### 4.2 Train

~3 minutes for 3 epochs. Watch `Validation ... Avg loss` fall between epochs.

In [ ]:
rc, train_time = H.run_container(
    "dli_train_lora", GPU,
    [(PTM, "/ptm:ro"), (SFT, "/data:ro"), (RUN, "/results")],
    ["cosmos-rl", "--config", "/results/config.toml", "/opt/cosmos_rl/tao_sft_example.py"],
    LOGS / "train.log")
assert rc == 0, "training failed — see the log above"

adapters = sorted(RUN.rglob("adapter_model.safetensors"))
print(f"\nadapters written: {len(adapters)}")
for a in adapters:
    print(f"  {a.parent.name:<12} {a.stat().st_size/1e6:.1f} MB   {a.parent}")

In [ ]:
# Validation loss per epoch — the training-side signal.
import re
losses = re.findall(r"avg_loss=([0-9.]+)", (LOGS / "train.log").read_text(errors="ignore"))
for i, l in enumerate(losses, 1):
    print(f"  epoch {i}: val avg_loss = {float(l):.5f}")
print("\nThe BEST epoch (lowest val loss) is what 'best' points at — later epochs often"
      "\noverfit on a dataset this small.")

---
### Part 5: Evaluate the Fine-Tuned Model

Same held-out clips, same metric — the only change is the LoRA adapter.

> Validation loss and task accuracy are **not** the same thing. A model can lower its loss
> by learning the output format while detecting nothing new. Always re-run the real metric.

In [ ]:
# Pick the adapter from the epoch with the LOWEST validation loss.
# Cosmos-RL writes output/best/best_score.json for this. Note its sibling
# `best/checkpoints` symlink points at a CONTAINER path (/results/...), so it
# dangles on the host — read the JSON and map the epoch name yourself.
def best_adapter(run_dir: Path) -> Path:
    score = next(run_dir.rglob("best/best_score.json"), None)
    if score:
        meta = json.loads(score.read_text())
        epoch = Path(meta["best_ckpt_abs_dir"]).name              # e.g. "epoch_2"
        hit = next(run_dir.rglob(f"safetensors/{epoch}/adapter_model.safetensors"), None)
        if hit:
            print(f"best epoch = {epoch}  ({meta['metric']}={meta['best_score']:.5f})")
            return hit.parent
    print("best_score.json not found — falling back to the last epoch")
    return sorted(run_dir.rglob("adapter_model.safetensors"))[-1].parent

best = best_adapter(RUN)
print(f"evaluating adapter: {best}\n")
tuned_metrics, tuned_time = run_eval("finetuned", lora_host=best)
H.show("Cosmos 3 Nano + LoRA", tuned_metrics)

---
### Part 6: Compare

In [ ]:
print(f"{'model':<26}{'macro-F1':>10}{'precision':>11}{'recall':>9}")
print("-" * 56)
for nm, mm in (("zero-shot", base_metrics), ("LoRA fine-tuned", tuned_metrics)):
    print(f"{nm:<26}{mm['macro_f1']:>10.3f}{mm['precision']:>11.3f}{mm['recall']:>9.3f}")
d = tuned_metrics["macro_f1"] - base_metrics["macro_f1"]
print(f"{'delta':<26}{d:>+10.3f}")

print("\nper-class detected (held-out clips)")
for c in sorted(set(base_metrics["per_class"]) | set(tuned_metrics["per_class"])):
    print(f"  {c:<20} {base_metrics['per_class'].get(c,'-'):>8}  ->  "
          f"{tuned_metrics['per_class'].get(c,'-'):>8}")

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
names = ["zero-shot", "LoRA"]
for i, k in enumerate(["macro_f1", "recall"]):
    vals = [base_metrics[k], tuned_metrics[k]]
    b = ax[i].bar(names, vals, color=["#8899a6", "#76b900"])
    ax[i].set_title({"macro_f1": "macro-F1", "recall": "recall"}[k])
    ax[i].set_ylim(0, 1.05); ax[i].bar_label(b, fmt="%.3f", padding=3)
    ax[i].spines[["top", "right"]].set_visible(False)
fig.suptitle("Cosmos 3 Nano on held-out traffic clips", y=1.02)
plt.tight_layout(); plt.savefig("results.png", dpi=110, bbox_inches="tight")
print("saved results.png"); plt.show()

#### What the numbers mean

**Precision stays at 1.000 and recall rises.** The base model was never *wrong* about the
anomalies it flagged — it simply missed most of them. Fine-tuning closed that gap without
introducing false alarms, which is the behaviour you want: at deployment scale, a few
percent of false positives on a mostly-normal stream buries operators in alerts.

**Look at the per-class row, not just the headline.** The class with the most training
windows improves the most. The class with the fewest barely moves — and that is the real
lesson of this notebook: *fine-tuning transfers where you gave it data, and nowhere else.*

**`speeding` and `collision` show no held-out numbers at all** because each has only one
source clip, which went to training. You cannot both train on an example and honestly test
on it.

#### Where this goes next

- **More data for the weak classes.** The per-class table is the specification for the SDG
  stage: generate more *distinct* source clips for whichever class failed to move. More
  windows cut from the same clip do not substitute for genuine variety.
- **Deploy the adapter.** It is ~15 MB and loads on top of the base model; the same
  checkpoint drives the VSS alert-verification pipeline later today.

---
### Part 7: Tear Down

Free the GPU and reclaim disk. FSDP training checkpoints are large; the adapter is not.

In [ ]:
subprocess.run(["docker", "rm", "-f", "dli_train_lora", "dli_eval_baseline", "dli_eval_finetuned"],
               capture_output=True)
ckpts = list(RUN.rglob("checkpoints"))
size = sum(f.stat().st_size for c in ckpts for f in c.rglob("*") if f.is_file()) / 1e9
print(f"training checkpoints: {size:.1f} GB in {len(ckpts)} dir(s)")
print("adapters kept:", [f"{a.stat().st_size/1e6:.0f} MB" for a in sorted(RUN.rglob('adapter_model.safetensors'))])
# Delete the large FSDP training checkpoints; the LoRA adapters are preserved.
import shutil
for c in ckpts:
    shutil.rmtree(c, ignore_errors=True)
print(f"reclaimed {size:.1f} GB — adapters remain in {RUN}")

---
### Review

In this notebook you covered:
- **Why post-training**, and how to tell a prompting ceiling from a model that is merely
  badly prompted - the base model here was precise but timid, which prompting could not fix
- **Dataset organisation for VLM fine-tuning**: cutting labelled windows from long video, and
  splitting on **clip identity** so near-duplicate frames never straddle train and test
- **macro-F1** as the metric for an imbalanced task, and why plain accuracy flatters a model
  that answers "no" to everything
- **A zero-shot baseline measured before training** - without it you cannot claim an
  improvement afterwards
- **LoRA post-training** with TAO + Cosmos-RL: the adapter rank, the spec keys that must be
  right, and why the artifact is 15 MB instead of 17 GB
- **Validation loss vs task accuracy**: the lowest-loss epoch is not automatically the most
  accurate checkpoint, so the real metric is re-run at the end
- **Per-class results**: fine-tuning transferred on the class with real training data and did
  nothing for the class with a single clip - which is the specification for what to generate
  next

**Next:** the same fine-tuned checkpoint is deployed behind VSS for alert verification, and
Part 2 of the day runs this whole workflow again through Agent Skills instead of cells.